# Tanager Mangrove Mapping - 01b Multispectral-Equivalent Preprocessing

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | July 2026 |

---

**Scope (Scenario B — journal publication only):** Load Tanager HDF5, convolve with Sentinel-2A SRF to produce multispectral-equivalent bands (B3, B4, B8, B11), compute reduced feature stack (NDMI, MVI, MNDWI, SAVI), save outputs to `data/processed_s2eq/`. REIP and EMI are excluded by design (not computable from S2 bandpasses).

**Relationship to Scenario A:** This notebook mirrors `01_preprocessing.ipynb` but replaces discrete-wavelength band extraction with SRF-weighted resampling. Pseudo-label generation logic (adaptive threshold, coastal candidate mask) is held constant between scenarios to isolate the effect of the feature stack.

## 0. Environment Setup

In [ ]:
# !pip install h5py scikit-image geopandas rasterio matplotlib openpyxl xarray scipy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
import json
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# ============================================================
# Project root and paths
# ============================================================
ROOT           = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROC      = ROOT / 'data' / 'processed'         # Scenario A (reference GeoTIFFs)
DATA_PROC_S2EQ = ROOT / 'data' / 'processed_s2eq'    # Scenario B outputs
DATA_SRF       = ROOT / 'data' / 'sentinel2_srf'
OUT_RESULTS    = ROOT / 'outputs' / 'results'
OUT_FIGURES    = ROOT / 'outputs' / 'figures'

for d in (DATA_PROC_S2EQ, OUT_RESULTS, OUT_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))

# ============================================================
# Reload src modules during development
# ============================================================
import src.preprocessing       as _pre
import src.spectral_resampling as _srf
importlib.reload(_pre)
importlib.reload(_srf)

from src.preprocessing import (
    load_hdf5,
    compute_coastal_candidate_mask,
    apply_adaptive_threshold,
)
from src.spectral_resampling import (
    S2_TARGET_BANDS,
    load_sentinel2_srf,
    resample_hyperspectral_to_s2,
    compute_scenario_b_indices,
    write_scenario_b_outputs,
)

# ============================================================
# Site registry (same as Scenario A)
# ============================================================
SITES = {
    'sangatta'   : '20250302_030003_92_4001',
    'gujarat'    : '20250311_061550_53_4001',
    'elsalvador' : '20250223_165546_32_4001',
    'belize'     : '20250824_171857_84_4001',
    'australia'  : '20250608_014315_58_4001',
}

SRF_FILE = DATA_SRF / 'COPE-GSEG-EOPG-TN-15-0007 - Sentinel-2 Spectral Response Functions 2024 - 4.0.xlsx'

print(f'ROOT           : {ROOT}')
print(f'Sites          : {list(SITES.keys())}')
print(f'SRF file       : {SRF_FILE.name}')
print(f'S2 bands used  : {list(S2_TARGET_BANDS.keys())} (B3, B4, B8, B11)')


## 1. Load Sentinel-2 Spectral Response Function

Source: ESA COPE-GSEG-EOPG-TN-15-0007 v4.0 (2024). Platform: Sentinel-2A.
SRF loaded once and reused across all sites.

In [ ]:
# ============================================================
# Load SRF -- one call, reused in the loop below
# ============================================================
print('Loading Sentinel-2A SRF...')
srf_dict = load_sentinel2_srf(SRF_FILE, platform='S2A')
print(f'\n  SRF loaded     : {len(srf_dict)} bands (green, red, nir, swir1)')


## 2. Per-Site Resampling and Index Computation

For each site:
1. Load full HDF5 (426 bands) via `load_hdf5()` from `src/preprocessing.py`
2. Convolve with S2A SRF -> 4 equivalent bands (B3, B4, B8, B11)
3. Compute NDMI, MVI, MNDWI, SAVI from resampled bands
4. Write to `data/processed_s2eq/{site_key}_s2eq_indices.tif`

CRS and transform inherited from Scenario A `*_bands.tif` to ensure pixel-perfect alignment.

In [ ]:
# ============================================================
# Main resampling loop -- all sites
# ============================================================
for site, scene_id in SITES.items():
    print(f'\n{"="*60}')
    print(f'  Site           : {site}  ({scene_id})')
    print(f'{"="*60}')

    h5_path       = DATA_RAW  / f'{site}_{scene_id}_ortho_sr_hdf5.h5'
    reference_tif = DATA_PROC / f'{site}_{scene_id}_bands.tif'
    site_key      = f'{site}_{scene_id}'

    # 1. Load full 426-band HDF5 spectrum
    #    Returns: reflectance (426, H, W), wavelengths (426,), crs, transform
    hdf5_data = load_hdf5(str(h5_path))

    # 2. Convolve with S2A SRF -> 4 equivalent bands
    resampled = resample_hyperspectral_to_s2(
        hdf5_data['reflectance'],
        hdf5_data['wavelengths'],
        srf_dict,
    )

    # 3. Compute Scenario B indices
    indices_b = compute_scenario_b_indices(resampled)

    # 4. Write outputs (inherits CRS + transform from Scenario A reference_tif)
    write_scenario_b_outputs(
        indices_b,
        reference_tif = reference_tif,
        out_dir       = DATA_PROC_S2EQ,
        site_key      = site_key,
    )


## 3. Sanity Check: Band Reflectance Comparison

Visual check: S2-equivalent band reflectance (Scenario B) vs discrete-wavelength extraction (Scenario A). Expected: values close but not identical (SRF averages over ~30-100 nm window; Scenario A uses single-band point extraction). If magnitudes differ by more than ~0.05 for water/vegetation, inspect resampling.

In [ ]:
# ============================================================
# Sanity check on Sangatta (training site)
# Compare mean band reflectance: Scenario A vs Scenario B
# ============================================================
SITE      = 'sangatta'
SCENE_ID  = SITES[SITE]
site_key  = f'{SITE}_{SCENE_ID}'

# Load Scenario A 5-band stack
ref_path = DATA_PROC / f'{site_key}_bands.tif'
with rasterio.open(ref_path) as src:
    bands_a = src.read()   # (5, H, W)

# Load Scenario B 4-band stack
s2eq_path = DATA_PROC_S2EQ / f'{site_key}_s2eq_indices.tif'
with rasterio.open(s2eq_path) as src:
    indices_b_arr = src.read()   # (4, H, W) -- NDMI, MVI, MNDWI, SAVI

# Report: mean index value per scenario
names_a = ['NDMI', 'MVI', 'MNDWI', 'SAVI', 'EMI']   # Scenario A order
names_b = ['NDMI', 'MVI', 'MNDWI', 'SAVI']           # Scenario B order

print('  Mean index values -- Sangatta:')
print(f'  {"Index":<8}  {"Scenario A":>12}  {"Scenario B":>12}  {"Diff":>8}')
print(f'  {"-"*46}')
for i, name in enumerate(names_b):
    idx_a = names_a.index(name)
    mean_a = float(np.nanmean(bands_a[idx_a]))
    mean_b = float(np.nanmean(indices_b_arr[i]))
    diff   = mean_b - mean_a
    print(f'  {name:<8}  {mean_a:>12.4f}  {mean_b:>12.4f}  {diff:>+8.4f}')


## 4. Adaptive Threshold and Pseudo-Label Generation (Scenario B)

Reuses `compute_coastal_candidate_mask` and `apply_adaptive_threshold` from `src/preprocessing.py` unchanged. Only input is the Scenario B index stack. This ensures pseudo-label generation logic is held constant between scenarios.

In [ ]:
# ============================================================
# Adaptive threshold + pseudo-label generation -- all sites
# Same logic as 01_preprocessing.ipynb Section 4
# ============================================================
all_thresholds_b = {}

for site, scene_id in SITES.items():
    print(f'\n{"="*60}')
    print(f'  Site           : {site}  ({scene_id})')
    print(f'{"="*60}')

    site_key  = f'{site}_{scene_id}'
    s2eq_path = DATA_PROC_S2EQ / f'{site_key}_s2eq_indices.tif'

    # Load Scenario B index stack from disk
    with rasterio.open(s2eq_path) as src:
        arr    = src.read().astype(np.float32)   # (4, H, W)
        data_b = {'transform': src.transform, 'crs': src.crs}

    # Reconstruct indices dict (band order: NDMI, MVI, MNDWI, SAVI)
    indices_b = {
        'NDMI'  : arr[0],
        'MVI'   : arr[1],
        'MNDWI' : arr[2],
        'SAVI'  : arr[3],
    }

    # Coastal candidate mask (adaptive Otsu -- same logic as Scenario A)
    candidate_mask = compute_coastal_candidate_mask(data_b, indices_b)

    # Adaptive threshold (MVI force_otsu -- same decision as Scenario A)
    thresholds = apply_adaptive_threshold(
        indices_b,
        scene_id=scene_id,
        candidate_mask=candidate_mask,
        force_otsu_indices=['MVI'],
    )

    # Save thresholds to disk
    thresh_path = OUT_RESULTS / f'thresholds_s2eq_{site}_{scene_id}.json'
    with open(thresh_path, 'w') as f:
        json.dump(
            {k: float(v) for k, v in thresholds.items()
             if v is not None and np.isfinite(v)},
            f, indent=2
        )
    print(f'  Thresholds saved : {thresh_path.name}')

    all_thresholds_b[site] = thresholds


## 5. Threshold Comparison: Scenario A vs Scenario B

Cross-check whether SRF resampling shifts adaptive thresholds significantly relative to Scenario A. Large shifts may indicate spectral differences between discrete extraction and SRF-averaged bands.

In [ ]:
# ============================================================
# Compare Scenario A vs B thresholds per site per index
# ============================================================
import pandas as pd

rows = []
for site, scene_id in SITES.items():
    # Load Scenario A thresholds (saved by 01_preprocessing)
    thresh_a_path = OUT_RESULTS / f'thresholds_{site}_{scene_id}.json'
    if not thresh_a_path.exists():
        print(f'  Scenario A thresholds not found for {site} -- skipping')
        continue
    with open(thresh_a_path) as f:
        thresh_a = json.load(f)

    thresh_b = all_thresholds_b.get(site, {})

    for idx in ['NDMI', 'MVI', 'MNDWI', 'SAVI']:
        t_a = thresh_a.get(idx, None)
        t_b = thresh_b.get(idx, None)
        rows.append({
            'site'        : site,
            'index'       : idx,
            'threshold_A' : round(t_a, 4) if t_a is not None else None,
            'threshold_B' : round(t_b, 4) if t_b is not None else None,
            'diff_B_A'    : round(t_b - t_a, 4) if (t_a and t_b) else None,
        })

df_thresh = pd.DataFrame(rows)
print('\n  Threshold comparison -- Scenario A vs B:')
print(df_thresh.to_string(index=False))
df_thresh.to_csv(OUT_RESULTS / 'threshold_comparison_A_vs_B.csv', index=False)
print('\n  Saved : threshold_comparison_A_vs_B.csv')
